# Faruq-v3 AF2+FFAB2 rFFT vs selected-DCT — Kaggle Decision
Decision-only notebook for the frozen efficiency stage.

Attach the three completed Stage-1 Kaggle outputs (for `AF2FFAB2FS`) and the three completed DCT Kaggle outputs (for `AF2FFADCTFS`) as Kaggle inputs. GPU is required because the frozen decision re-benchmarks adapter and full-model latency.

This notebook does not train and does not access locked test.


In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
SEEDS=(42,123,2026)
def one(pattern):
    hits=sorted(INPUT.rglob(pattern))
    if len(hits)!=1: raise FileNotFoundError(f'Harus ada tepat satu {pattern}; ditemukan {hits}')
    return hits[0]
RFFT=[one(f'AF2FFAB2FS_seed{s}_result.json') for s in SEEDS]
DCT=[one(f'AF2FFADCTFS_seed{s}_result.json') for s in SEEDS]
print('INPUT RESULT PREFLIGHT PASS')
for p in RFFT+DCT: print(p)


In [ ]:
import hashlib,importlib,json,os,shutil,subprocess,sys,time,torch
BRANCH='codex/af2-ffab2-from-start-dct'
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan Kaggle GPU untuk efficiency benchmark.')
os.chdir(WORK); REPO=WORK/'coffee-bean-detection'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('COMMIT:',COMMIT)
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
# Kaggle-mounted result JSONs contain the original /kaggle/working checkpoint path.
# Rebind each result to the mounted best.pt with the exact recorded SHA256 before benchmarking.
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
best_candidates=sorted(INPUT.rglob('best.pt'))
if not best_candidates: raise FileNotFoundError('Tidak ada best.pt pada attached Kaggle outputs.')
sha_index={}
for p in best_candidates:
    digest=sha256(p); sha_index.setdefault(digest,[]).append(p)
NORMALIZED=WORK/'normalized_dct_decision_inputs'; NORMALIZED.mkdir(exist_ok=True)
def normalize(result_path):
    payload=json.loads(result_path.read_text(encoding='utf-8'))
    digest=payload['checkpoint_sha256']
    matches=sha_index.get(digest,[])
    if len(matches)!=1: raise RuntimeError(f'Checkpoint SHA {digest} harus cocok tepat satu best.pt; ditemukan {matches}')
    payload['checkpoint']=str(matches[0])
    target=NORMALIZED/result_path.name
    target.write_text(json.dumps(payload,indent=2)+'\n',encoding='utf-8')
    return target
RFFT_N=[normalize(p) for p in RFFT]
DCT_N=[normalize(p) for p in DCT]
print('CHECKPOINT REBIND PASS')
for p in RFFT_N+DCT_N: print(p)


In [ ]:
# Validate pair contracts before the frozen efficiency decision.
for seed,r,d in zip(SEEDS,RFFT_N,DCT_N):
    pr=json.loads(r.read_text(encoding='utf-8')); pd=json.loads(d.read_text(encoding='utf-8'))
    assert pr['arm']=='AF2FFAB2FS' and pd['arm']=='AF2FFADCTFS'
    assert pr['seed']==seed and pd['seed']==seed
    assert pr['initial_d0_checkpoint_sha256']==pd['initial_d0_checkpoint_sha256']
    assert pr['evaluation_split']=='val' and pd['evaluation_split']=='val'
    assert pr['test_images_accessed'] is False and pd['test_images_accessed'] is False
print('THREE-SEED DCT CONTRACT PASS')


In [ ]:
SUMMARY=WORK/'af2_ffab2_dct_efficiency_decision.json'
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffa_dct_decision',
     '--rfft',*[str(p) for p in RFFT_N],'--dct',*[str(p) for p in DCT_N],'--output',str(SUMMARY),'--device','0']
subprocess.run(cmd,cwd=REPO,check=True)
decision=json.loads(SUMMARY.read_text(encoding='utf-8'))
assert decision['test_opened'] is False
print('\nDCT DECISION:',decision['decision'])
print('NEXT:',decision['next'])
print('CRITERIA:',json.dumps(decision['criteria'],indent=2))
print('EFFICIENCY:',json.dumps(decision['efficiency'],indent=2))
print('AGGREGATE:',json.dumps(decision['aggregate'],indent=2))
meta={'branch':BRANCH,'commit':COMMIT,'evaluation_split':'val','test_images_accessed':False,'decision':decision['decision'],'next':decision['next']}
(WORK/'af2_ffab2_dct_efficiency_decision_meta.json').write_text(json.dumps(meta,indent=2)+'\n',encoding='utf-8')
print('\nOUTPUT:',SUMMARY)
print('Locked test remains closed.')
